# 05_6 — Feature Engineering (individual vs. agrupado) + EXPORT para o Narrador (05_5)

**Reescrita do `05_6_Analise_Pareada_Clima_Atraso.ipynb`.** Duas entregas:

1. **Estudo de granularidade** — compara features climáticas **individuais** (leitura
   instantânea via `merge_asof`) vs. **agrupadas** (agregação por janela) de forma
   metodologicamente honesta.
2. **Exportação de produção** — treina e salva os **dois modelos binários pareados**
   (`modelo_par_clima` e `modelo_par_atraso`) no formato EXATO que o
   `05_5_Narrador_Natural.ipynb` carrega (mesmas chaves de metadados e mesmo *namespace*
   de features), para você "plugar" direto no arquivo de linguagem natural.

## Contrato de saída consumido pelo 05_5 (Narrador)
O narrador (`05_5`) faz `joblib.load` de `modelo_par_clima.joblib`/`modelo_par_atraso.joblib`
e lê `*_meta.json` esperando as chaves: `janela`, `dist_aeroporto`, `features_lgbm`,
`macro_f1_bal`, `macro_f1_nat`. Na inferência ele reconstrói as features com
`construir_features()` (namespace: `temperature_2m`, `relative_humidity_2m`,
`wind_speed_10m`, `surface_pressure`, `{c}_lag`, `diff_{c}`, `voos_cia_*`, `voos_orig_*`,
`voos_linha_*`, `voos_sev_*`, `qtd_voos_previstos`, `qtd_empresas_aereas`,
`atraso_medio_min`, `atraso_max_min`, `lat`, `lng`, `dist_aeroporto_km`, `hora`, `mes`,
`dia_semana`, `hora_sin/cos`, `mes_sin/cos`, `dia_semana_sin/cos`). Portanto a Seção 8
(produção) usa **exatamente esses nomes** e janela de **1h**, e passa features
**numéricas** (não `category`), para não haver *train/serve skew*.

## Correção metodológica central (verificada)
> A resolução do clima é **horária**. Com janela de 1h, `individual ≡ agrupado`; e
> `min/max/std` por janela de 1h são **degenerados**. Por isso o **estudo** de granularidade
> usa janela **agrupada de 4h** (agrega ≥1 leitura) e agregações válidas (rolling
> multi-horizonte por TEMPO + razões físicas), enquanto a **produção** usa 1h (a janela
> operacional do narrador — onde agrupado=individual, o que é correto para produção).

## Mudanças em relação ao original (todas verificadas adversarialmente)
- **[BUG CORRIGIDO] Comparação pareada real:** ambos os regimes agora são em **nível de
  evento** (cada evento recebe as features da sua janela / instantâneas), com `dataset_type`
  e corte por `request_dt` idênticos. Antes o regime agrupado era por janela e não tinha
  rótulo → `KeyError`.
- **share por CIA vetorizado** (evita bug do `groupby.apply` no pandas ≥2.2).
- **rolling por TEMPO** (janela `3h/6h/24h` sobre `time` como DatetimeIndex; reatribuição
  posicional), não por nº de linhas.
- **detecção dinâmica** das variáveis físicas (não lista hardcoded silenciosa).
- **anti-vazamento Análise B:** exclui `atraso_*`, `sev_*`, `STATUS_ATRASO`, `*_REAL` **e**
  `qtd_voos`/contagens da classe positiva.
- **PR-AUC**, **split por timestamp**, **refit em treino+val (R3)**.
- **setseed materializado** (`duckdb.execute(...).fetchall()`).
- **remoção de código morto** (`StandardScaler`, `USE_DIST` ocioso, etc.).

> Notebook para Colab (GCS + Drive). **Não executado aqui** — rode no seu ambiente. Nomes de
> colunas do CSV de clima seguem o contrato do 05_5; ajuste os marcadores `<-- ajuste` se diferirem.


## 1. Setup e configuração


In [ ]:
!pip install -q gcsfs duckdb lightgbm shap scikit-learn pandas pyarrow h3

import numpy as np
import pandas as pd
import duckdb
import lightgbm as lgb
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
import shap
import matplotlib.pyplot as plt
import json, os
from sklearn.metrics import classification_report, f1_score, average_precision_score
# StandardScaler REMOVIDO (import morto no original; árvores não precisam de padronização).
try:
    import h3
except Exception:
    h3 = None   # só necessário se USE_DIST/COMPUTAR_ESPACIAL


In [ ]:
# ------------------- Configuração -------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Janelas: ESTUDO usa 4h (> resolução horária -> comparação válida); PRODUÇÃO usa 1h
# (janela operacional do narrador; a 1h agrupado=individual, o que é o correto p/ produção).
JANELA_AGRUPADO = '4h'
JANELA_PROD     = '1h'

USE_DIST          = False   # distância NÃO é mais feature do modelo: virou fator de "esforço" do
                            # motorista no 05_5 (entra na decisão/narração, fora do modelo/SHAP).
                            # Como era servida como constante fixa, aparecia no SHAP com valor que
                            # nunca variava (enganoso). Ver dist_aeroporto_km em COLS_ESPACIAIS.
COMPUTAR_ESPACIAL = False   # lat/lng crus como feature? (risco de vazamento espacial)

# scale_pos_weight do modelo de ATRASO (amplificação do sinal): >1 aumenta o recall da classe
# "atraso" à custa de precisão. Padrões de atraso, mesmo esporádicos, ficam mais fáceis de captar.
# Ajustável; None desliga (comportamento balanceado do sampler). Só o modelo de atraso usa.
PESO_ATRASO = 2.0

# Flag central da tensão rigor x compatibilidade do narrador:
# O CSV de voos é "03_voos_atrasados_sbpa.csv" = SOMENTE voos atrasados. Logo TODA feature
# derivada de voos (qtd_voos_previstos, voos_cia_*, atraso_medio_min, sev_*) é PÓS-FATO para
# prever atraso -> vazamento. Padrão RIGOROSO = False (modelo de atraso usa só clima+tempo).
# True reproduz o comportamento ORIGINAL do narrador (features de voo enriquecidas).
INCLUIR_FEATURES_POSFATO_ATRASO = False

# Escores compostos de UTCI: os eventos DS_CLIMA foram gerados a partir deles -> vazam.
def eh_score_composto(col: str) -> bool:
    c = col.lower()
    return any(p in c for p in ('utci', 'discomfort', 'stress', 'score', 'severidade',
                                'thermal', 'conforto', 'desconforto'))

# Variáveis pós-fato / definicionais de atraso (vazamento na Análise B / modelo de atraso).
def eh_vazamento_atraso(col: str) -> bool:
    c = col.lower()
    return any(p in c for p in ('atraso', 'status_atraso', '_real', 'delay',
                                'minutos_atraso', 'sev_', 'qtd_voos', 'voos_'))

# Colunas espaciais mantidas fora das features (vazamento espacial). dist_aeroporto_km entra
# aqui de propósito (belt-and-suspenders): garante exclusão do modelo/SHAP mesmo se a coluna
# for criada. A distância passou a ser tratada só como "esforço" do motorista no 05_5.
COLS_ESPACIAIS = ['lat', 'lng', 'origin_h3', 'densidade_h3', 'dist_aeroporto_km']

# Namespace de voos (idêntico ao _agregar_voos do 05_5, para compatibilidade de produção).
TOP_CIAS    = ['TAM', 'AZU', 'GLO', 'TAP', 'ARG']
TOP_ORIGENS = ['SBGR', 'SBSP', 'SBKP', 'SBCF', 'SBBR']
# Variáveis climáticas com lag/diff no contrato do narrador:
COLS_LAG_PROD = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'surface_pressure']

AER_LAT, AER_LNG = -29.9939, -51.1711   # SBPA (Salgado Filho)

DRIVE_BASE    = '/content/drive/MyDrive/DOUTORADO'
CACHE_PARQUET = f'{DRIVE_BASE}/cache_amostra_05_6.parquet'
# Caminhos de EXPORT lidos pelo 05_5:
EXP_CLIMA   = f'{DRIVE_BASE}/modelo_par_clima.joblib'
EXP_CLIMA_M = f'{DRIVE_BASE}/modelo_par_clima_meta.json'
EXP_ATRASO  = f'{DRIVE_BASE}/modelo_par_atraso.joblib'
EXP_ATRASO_M= f'{DRIVE_BASE}/modelo_par_atraso_meta.json'

# --- Climatologia de atraso na CHEGADA (feature PROSPECTIVA, sem vazamento) ---
# Fonte: 01_todos_voos_consolidados.csv (VRA/ANAC) = TODOS os voos de 2025 (pontuais +
# atrasados), ao contrário do 03_voos_atrasados_sbpa.csv (só atrasados => pós-fato). Com o
# denominador (voos pontuais) dá para calcular a TAXA histórica de atraso por (dia_semana,
# hora): um PRIOR disponível ANTES do fato (não é pós-fato), que pode entrar legitimamente no
# modelo de atraso. A tabela é exportada para o 05_5 reproduzir a feature no serve (sem skew).
CONSOLIDADO_VOOS  = f'{DRIVE_BASE}/01_PRE_GERACAO/01_todos_voos_consolidados.csv'  # <-- ajuste o caminho
TZ_VRA            = 'America/Sao_Paulo'   # fuso das colunas do VRA; use None se já estiver em UTC  # <-- ajuste
LIMIAR_ATRASO_MIN = 15                    # atraso na chegada (min) para contar como "atrasado"
PSEUDO_CONTAGEM_CLIM = 20                 # força do prior (voos-equivalentes) no shrinkage empírico-Bayes:
                                          # células (dia_semana,hora) com poucos voos são puxadas p/ a taxa global
EXP_CLIMATOLOGIA  = f'{DRIVE_BASE}/climatologia_atraso_sbpa.csv'   # tabela (dia_semana,hora) lida pelo 05_5
EXP_REPUTACAO_SLOT = f'{DRIVE_BASE}/reputacao_slot_sbpa.csv'       # piores CIAs/origens por slot (contexto textual p/ 05_5)

print('Config OK | estudo=%s | produção=%s | posfato_atraso=%s'
      % (JANELA_AGRUPADO, JANELA_PROD, INCLUIR_FEATURES_POSFATO_ATRASO))

In [ ]:
# --- Conexão com Google Drive + Google Cloud Storage (Colab) ---
# Padrão idêntico ao 05_4/05_6 originais (funciona sem HMAC/httpfs): autentica no GCP,
# cria o gcsfs e o REGISTRA na conexão DuckDB (assim `read_parquet('gs://...')` funciona
# em bucket privado). A mesma conexão `con` guarda o `setseed` materializado (Seção 2).
from google.colab import auth, drive
import gcsfs, shutil

auth.authenticate_user()
project_id  = 'doutorado-501917'   # projeto GCP do doutorado
bucket_name = '2025_rides'         # bucket dos eventos de corrida (Parquet)
fs = gcsfs.GCSFileSystem(project=project_id)

con = duckdb.connect()                            # conexão PERSISTENTE (mantém o seed)
con.register_filesystem(fs)                       # habilita leitura gs:// via gcsfs
con.execute("SELECT setseed(0.42)").fetchall()    # materializa a semente (reprodutibilidade)
try:
    duckdb.register_filesystem(fs)                # também no default, por segurança
except Exception:
    pass

drive.mount('/content/drive')

# Os CSVs vivem em subpastas do Drive; copia para /content e lê localmente (rápido/robusto).
base_dir     = f'{DRIVE_BASE}/003_DADOS_SINTETICOS/arquivos_base'
shutil.copy(f'{base_dir}/dados_meteorologicos_utci_horario.csv', './')
shutil.copy(f'{base_dir}/DADOS_AEROPORTO/03_voos_atrasados_sbpa.csv', './')

df_clima = pd.read_csv('/content/dados_meteorologicos_utci_horario.csv')
df_voos  = pd.read_csv('/content/03_voos_atrasados_sbpa.csv', sep=';')
df_clima['time'] = pd.to_datetime(df_clima['time'], utc=True, errors='coerce')
df_clima = df_clima.dropna(subset=['time']).sort_values('time').reset_index(drop=True)
print('Clima:', df_clima.shape, '| Voos:', df_voos.shape)
print('Colunas de voos:', list(df_voos.columns))


In [ ]:
# --- Prior prospectivo de atraso na CHEGADA em SBPA (climatologia + reputação por slot) ---
# Lê o VRA consolidado (TODOS os voos de 2025, pontuais + atrasados) e devolve uma tabela
# indexada por (dia_semana, hora) EM UTC com SEIS features PROSPECTIVAS (disponíveis antes
# do fato -> não vazam), que passam a alimentar o modelo de atraso:
#   Climatologia (volume/severidade do slot):
#     - taxa_atraso_hist        : P(chegada atrasada > limiar | dia_semana, hora)
#     - atraso_medio_hist       : atraso médio de chegada (min), com piso em 0
#     - chegadas_previstas_hist : nº médio de chegadas programadas na janela (contexto de volume)
#   Reputação da companhia (3 níveis), AGREGADA ao slot pela composição histórica de escala:
#     - rep_cia_slot        : reputação média das CIAs que costumam pousar naquele slot
#     - rep_cia_origem_slot : reputação média dos pares (CIA, origem) daquele slot (situacional)
#     - rep_origem_slot     : reputação média das origens daquele slot
# ENQUADRAMENTO ANTI-VAZAMENTO: o narrador NÃO sabe qual voo específico vai chegar no horário
# consultado — só conhece o slot (dia_semana, hora). Por isso features por-voo (voos_cia_*,
# voos_orig_*) são pós-fato. Aqui as reputações das ENTIDADES são estimadas no histórico e
# depois AGREGADAS ao slot pela composição de escala típica daquele horário -> prospectivo.
#
# SHRINKAGE (empirical Bayes) em toda taxa: (n*obs + K*alvo)/(n+K), K = pseudo. Células/entidades
# com muitos voos quase não mudam; as raras deixam de exibir extremos ruidosos (0%/100%) e
# ficam próximas do alvo. A reputação do PAR (nível 2) encolhe hierarquicamente rumo à
# reputação da própria CIA (nível 1); CIA e origem encolhem rumo à taxa global.
# chegadas_previstas_hist é VOLUME (não taxa) -> não recebe shrinkage.
#
# corte_utc (opcional): se informado, mantém só voos com CHEGADA_PREVISTA (UTC) <= corte, para
# ajustar o prior SOMENTE com o período de treino (evita vazamento temporal na avaliação). Como
# o modelo exportado também é treinado só nesse período, o mesmo prior serve treino e produção
# sem skew — por isso NÃO usamos dois priors distintos.
def construir_prior(caminho, tz_vra=TZ_VRA, limiar_min=LIMIAR_ATRASO_MIN,
                    pseudo=PSEUDO_CONTAGEM_CLIM, corte_utc=None,
                    topk_ctx=3, min_n_ctx=5):
    usecols = ['ICAO_EMPRESA_AEREA', 'ICAO_AERODROMO_ORIGEM', 'ICAO_AERODROMO_DESTINO',
               'CHEGADA_PREVISTA', 'CHEGADA_REAL', 'SITUACAO_VOO']
    dfv = pd.read_csv(caminho, sep=';', usecols=usecols, dtype=str, encoding='utf-8')
    dfv = dfv[(dfv['ICAO_AERODROMO_DESTINO'] == 'SBPA') &
              (dfv['SITUACAO_VOO'] == 'REALIZADO')].copy()
    prev = pd.to_datetime(dfv['CHEGADA_PREVISTA'], errors='coerce')
    real = pd.to_datetime(dfv['CHEGADA_REAL'],     errors='coerce')
    dfv = dfv.assign(_prev=prev, _real=real).dropna(subset=['_prev', '_real'])
    # Local (VRA) -> UTC, para alinhar com os eventos. (Brasil sem horário de verão desde 2019.)
    if tz_vra:
        prev_utc = (dfv['_prev'].dt.tz_localize(tz_vra, ambiguous='NaT',
                                                nonexistent='shift_forward')
                                .dt.tz_convert('UTC'))
    else:
        prev_utc = dfv['_prev'].dt.tz_localize('UTC')
    dfv = dfv.assign(_prev_utc=prev_utc).dropna(subset=['_prev_utc'])
    # Corte temporal (train-only): mantém só voos cuja CHEGADA_PREVISTA (UTC) <= corte.
    n_total = len(dfv)
    if corte_utc is not None:
        corte = pd.Timestamp(corte_utc)
        corte = corte.tz_localize('UTC') if corte.tzinfo is None else corte.tz_convert('UTC')
        dfv = dfv[dfv['_prev_utc'] <= corte].copy()
    atraso_min = (dfv['_real'] - dfv['_prev']).dt.total_seconds() / 60.0   # diff local (invariante ao fuso)
    dfv['_atrasado']   = (atraso_min > limiar_min).astype(int)
    dfv['_atraso_pos'] = atraso_min.clip(lower=0)
    dfv['dia_semana']  = dfv['_prev_utc'].dt.dayofweek
    dfv['hora']        = dfv['_prev_utc'].dt.hour
    dfv['_data']       = dfv['_prev_utc'].dt.date
    dfv['_cia']  = dfv['ICAO_EMPRESA_AEREA'].astype(str).str.upper()
    dfv['_orig'] = dfv['ICAO_AERODROMO_ORIGEM'].astype(str).str.upper()

    # Priors globais (alvos do shrinkage)
    taxa_global         = float(dfv['_atrasado'].mean())
    atraso_medio_global = float(dfv['_atraso_pos'].mean())

    def _shrink(soma, n, alvo):   # empirical Bayes: (soma + K*alvo)/(n+K), com obs = soma/n
        return (soma + pseudo * alvo) / (n + pseudo)

    # ---- Climatologia por slot (dia_semana, hora) ----
    g = dfv.groupby(['dia_semana', 'hora'])
    clim = g.agg(_soma_atr=('_atrasado', 'sum'),
                 _soma_min=('_atraso_pos', 'sum'),
                 _n_voos=('_atrasado', 'size'),
                 _n_datas=('_data', 'nunique')).reset_index()
    n = clim['_n_voos']
    clim['taxa_atraso_hist']        = _shrink(clim['_soma_atr'], n, taxa_global)
    clim['atraso_medio_hist']       = _shrink(clim['_soma_min'], n, atraso_medio_global)
    clim['chegadas_previstas_hist'] = clim['_n_voos'] / clim['_n_datas'].clip(lower=1)

    # ---- Reputações por ENTIDADE (empirical Bayes) ----
    rc = dfv.groupby('_cia')['_atrasado'].agg(['sum', 'size'])   # Nível 1: CIA -> global
    rep_cia  = _shrink(rc['sum'], rc['size'], taxa_global).to_dict()
    ro = dfv.groupby('_orig')['_atrasado'].agg(['sum', 'size'])  # Nível 3: origem -> global
    rep_orig = _shrink(ro['sum'], ro['size'], taxa_global).to_dict()
    rco = dfv.groupby(['_cia', '_orig'])['_atrasado'].agg(['sum', 'size']).reset_index()
    rco['_alvo'] = rco['_cia'].map(rep_cia).fillna(taxa_global)   # Nível 2: par -> shrink p/ CIA
    rco['_rep']  = (rco['sum'] + pseudo * rco['_alvo']) / (rco['size'] + pseudo)
    rep_par = {(c, o): r for c, o, r in zip(rco['_cia'], rco['_orig'], rco['_rep'])}

    # ---- Agregação ao SLOT pela composição histórica de escala ----
    # média por voo no slot == média ponderada pela contagem de voos de cada entidade no slot.
    dfv['_rep_cia']  = dfv['_cia'].map(rep_cia).fillna(taxa_global)
    dfv['_rep_orig'] = dfv['_orig'].map(rep_orig).fillna(taxa_global)
    dfv['_rep_par']  = [rep_par.get((c, o), taxa_global) for c, o in zip(dfv['_cia'], dfv['_orig'])]
    rep_slot = dfv.groupby(['dia_semana', 'hora']).agg(
        rep_cia_slot=('_rep_cia', 'mean'),
        rep_cia_origem_slot=('_rep_par', 'mean'),
        rep_origem_slot=('_rep_orig', 'mean')).reset_index()

    prior = (clim[['dia_semana', 'hora', 'taxa_atraso_hist', 'atraso_medio_hist',
                   'chegadas_previstas_hist']]
             .merge(rep_slot, on=['dia_semana', 'hora'], how='left'))

    fallback = {'taxa_atraso_hist': taxa_global, 'atraso_medio_hist': atraso_medio_global,
                'chegadas_previstas_hist': 0.0, 'rep_cia_slot': taxa_global,
                'rep_cia_origem_slot': taxa_global, 'rep_origem_slot': taxa_global}

    # ---- Contexto textual p/ o 05_5: piores pares (CIA, origem) por slot (NÃO entra no modelo) ----
    grp = dfv.groupby(['dia_semana', 'hora', '_cia', '_orig']).size().reset_index(name='_n')
    tot = dfv.groupby(['dia_semana', 'hora']).size().rename('_tot').reset_index()
    grp = grp.merge(tot, on=['dia_semana', 'hora'])
    grp['share'] = grp['_n'] / grp['_tot']
    grp['rep']   = [rep_par.get((c, o), taxa_global) for c, o in zip(grp['_cia'], grp['_orig'])]
    grp = grp[grp['_n'] >= min_n_ctx]
    rep_ctx = (grp.sort_values(['dia_semana', 'hora', 'rep'], ascending=[True, True, False])
                  .groupby(['dia_semana', 'hora']).head(topk_ctx)
                  [['dia_semana', 'hora', '_cia', '_orig', 'rep', 'share']]
                  .rename(columns={'_cia': 'cia', '_orig': 'origem'})
                  .reset_index(drop=True))
    print('  construir_prior: %d voos usados%s | taxa global = %.1f%%'
          % (len(dfv), '' if corte_utc is None else ' de %d (train-only, corte=%s)' % (n_total, corte),
             100 * taxa_global))
    return prior, fallback, rep_ctx

# Nomes das features do prior (climatologia + reputação) — usados na seleção e no check de namespace.
COLS_CLIMATOLOGIA = ['taxa_atraso_hist', 'atraso_medio_hist', 'chegadas_previstas_hist',
                     'rep_cia_slot', 'rep_cia_origem_slot', 'rep_origem_slot']

print('construir_prior() definida | COLS_CLIMATOLOGIA =', COLS_CLIMATOLOGIA)
print('(o prior é construído na Seção 8, após CORTE_PROD, para ser train-only — sem vazamento)')

## 2. Amostra estratificada (cache) — padrão eficiente do original

Amostragem balanceada por *bucket* temporal × classe no DuckDB, com cache em Parquet.
`setseed` é **materializado** com `.fetchall()` (senão a relação lazy do DuckDB não executa
o seed e o `RANDOM()` fica não determinístico).


In [ ]:
# Globs reais dos eventos de corrida no GCS (mesmo layout do 05_4): V6_3 .. V6_7.
GCS_GLOBS = [f'gs://{bucket_name}/outputs_simulation_V6_{i}/trips_log/_staging/**/*.parquet'
             for i in range(3, 8)]
inicio_ts, fim_ts = 1735699200, 1767235200
N_BUCKETS, POR_BUCKET = 100, 300

if os.path.exists(CACHE_PARQUET):
    df_comb = pd.read_parquet(CACHE_PARQUET)
    print('Amostra lida do cache:', df_comb.shape, '-- GCS/DuckDB pulado.')
else:
    arquivos = []
    for g in GCS_GLOBS:
        arquivos.extend([f'gs://{a}' for a in fs.glob(g)])
    print(f'Total de {len(arquivos):,} arquivos Parquet no GCS.')
    files_sql_array = ", ".join([f"'{a}'" for a in arquivos])
    # `con` já foi criado na Seção 1 (com fs registrado e setseed materializado).
    query = f"""
    WITH base AS (
        SELECT request_ts, event_name, origin_h3,
            CASE
                WHEN UPPER(event_name) LIKE '%ATRASADO%'   THEN 'DS_VOO'
                WHEN UPPER(event_name) LIKE '%SEVERIDADE%' THEN 'DS_CLIMA'
                WHEN event_name IS NULL                    THEN 'NULO'
                ELSE 'DS_OUTROS'
            END AS dataset_type,
            LEAST({N_BUCKETS - 1},
                  CAST((request_ts - {inicio_ts}) * {N_BUCKETS}.0
                       / ({fim_ts} - {inicio_ts}) AS INTEGER)) AS bucket
        FROM read_parquet([{files_sql_array}], hive_partitioning=true)
        WHERE request_ts >= {inicio_ts} AND request_ts < {fim_ts}
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY dataset_type, bucket
                                     ORDER BY RANDOM()) AS rn
        FROM base WHERE dataset_type <> 'NULO'
    )
    SELECT request_ts, event_name, origin_h3, dataset_type
    FROM ranked WHERE rn <= {POR_BUCKET}
    """
    df_comb = con.execute(query).df()
    df_comb.to_parquet(CACHE_PARQUET)
    print('Amostra materializada e cacheada:', df_comb.shape)

df_comb['request_dt'] = pd.to_datetime(df_comb['request_ts'], unit='s', utc=True)
print(df_comb['dataset_type'].value_counts())

# RESSALVA (L2): a query equaliza classes por bucket -> o teste NÃO é populacional (<1% real).
DS_VOO    = df_comb[df_comb['dataset_type'] == 'DS_VOO'].copy()
DS_CLIMA  = df_comb[df_comb['dataset_type'] == 'DS_CLIMA'].copy()
DS_OUTROS = df_comb[df_comb['dataset_type'] == 'DS_OUTROS'].copy()


## 3. Feature engineering do ESTUDO (individual vs. agrupado)

Rolling multi-horizonte por TEMPO (3h/6h/24h) + razões físicas, sobre a série HORÁRIA,
ANTES de agregar/mesclar. Detecção dinâmica das variáveis físicas (evita falha silenciosa).


In [ ]:
def _detectar_vars_fisicas(dfc):
    """Colunas numéricas que NÃO são escore composto (candidatas a variáveis físicas)."""
    num = dfc.select_dtypes('number').columns
    vf = [c for c in num if not eh_score_composto(c)]
    return vf

def enriquecer_clima_horario(dfc):
    """Rolling por TEMPO (3h/6h/24h) + deltas + razão física, sobre a série horária."""
    dfc = dfc.sort_values('time').reset_index(drop=True).copy()
    vars_fisicas = _detectar_vars_fisicas(dfc)
    if not vars_fisicas:
        print('AVISO: nenhuma variável física detectada — enriquecimento vazio!')
    horizontes = {'3h': '3h', '6h': '6h', '24h': '24h'}
    # Rolling por TEMPO exige DatetimeIndex e é operação de DataFrame (o parâmetro on=
    # NÃO existe em Series.rolling). Fazemos o rolling sobre uma cópia indexada por 'time'
    # e reatribuímos por POSIÇÃO (.to_numpy()) — dfc já está ordenado e com índice resetado,
    # então a ordem posicional coincide com a da cópia indexada.
    dfc_t = dfc.set_index('time')
    for v in vars_fisicas:
        for nome, win in horizontes.items():
            dfc[f'{v}_media_{nome}'] = dfc_t[v].rolling(win).mean().to_numpy()
    # deltas temporais (shift posicional, ordem já garantida acima)
    for v in vars_fisicas:
        dfc[f'{v}_delta_3h'] = dfc[v] - dfc[v].shift(3)
    # razão física ilustrativa (ajuste aos nomes reais do seu CSV)
    if 'wind_speed_10m' in dfc.columns and 'temperature_2m' in dfc.columns:
        dfc['razao_vento_temp'] = dfc['wind_speed_10m'] / (dfc['temperature_2m'].abs() + 1e-3)
    print('Variáveis físicas enriquecidas:', vars_fisicas)
    return dfc

df_clima_enriq = enriquecer_clima_horario(df_clima)


In [ ]:
def construir_base_agrupado_estudo(window_size):
    """ESTUDO: base em NÍVEL DE EVENTO com features climáticas AGREGADAS por janela.
    Cada evento (DS_OUTROS/DS_CLIMA/DS_VOO) recebe as features da janela em que cai —
    mantendo dataset_type e request_dt para rótulo e corte temporal."""
    dfc = df_clima_enriq.copy()
    dfc['time_window'] = dfc['time'].dt.floor(window_size)
    clima_agg = dfc.drop(columns=['time']).groupby('time_window').mean(numeric_only=True)
    clima_agg = clima_agg.sort_index()
    # lags/diffs em nível de JANELA (defasagem de 1 janela)
    for c in list(clima_agg.columns):
        clima_agg[f'{c}_lag']  = clima_agg[c].shift(1)
        clima_agg[f'diff_{c}'] = clima_agg[c] - clima_agg[c].shift(1)
    clima_agg = clima_agg.reset_index()

    ev = pd.concat([DS_OUTROS, DS_CLIMA, DS_VOO], ignore_index=True) \
           .sort_values('request_dt').reset_index(drop=True)
    ev['time_window'] = ev['request_dt'].dt.floor(window_size)
    base = ev.merge(clima_agg, on='time_window', how='left')
    _add_temporais(base, 'request_dt')
    return base

def construir_base_individual_estudo(tol='1h'):
    """ESTUDO: base em NÍVEL DE EVENTO com a leitura climática INSTANTÂNEA (merge_asof
    backward): para cada evento, a leitura horária mais recente ANTES dele."""
    ev = pd.concat([DS_OUTROS, DS_CLIMA, DS_VOO], ignore_index=True) \
           .sort_values('request_dt').reset_index(drop=True)
    clima = df_clima_enriq.sort_values('time').reset_index(drop=True)
    base = pd.merge_asof(ev, clima, left_on='request_dt', right_on='time',
                         direction='backward', tolerance=pd.Timedelta(tol))
    _add_temporais(base, 'request_dt')
    return base

def _add_temporais(base, coluna_dt):
    """Adiciona hora/mes/dia_semana (numéricos) + codificação cíclica sin/cos.
    Escolha: cíclica (sin/cos) em vez de categórica nativa, para casar com o narrador
    (que passa features numéricas na inferência)."""
    dt = base[coluna_dt].dt
    base['hora'] = dt.hour; base['mes'] = dt.month; base['dia_semana'] = dt.dayofweek
    for col, period in [('hora', 24), ('mes', 12), ('dia_semana', 7)]:
        base[f'{col}_sin'] = np.sin(2*np.pi * base[col] / period)
        base[f'{col}_cos'] = np.cos(2*np.pi * base[col] / period)
    return base


## 4. Protocolo justo de comparação (corte fixo, split por timestamp, refit, PR-AUC)


In [ ]:
COLS_META = {'y', 'time', 'time_window', 'request_dt', 'request_ts', 'event_name',
             'dataset_type', 'bucket'}

def selecionar_features(df, cols_excluir):
    proibidas = set(cols_excluir) | COLS_META | set(COLS_ESPACIAIS)
    if USE_DIST:
        proibidas.discard('dist_aeroporto_km')   # dist é permitido se USE_DIST
    # Filtra também colunas não-numéricas (object, string, etc.) para garantir
    # compatibilidade com LightGBM nos dois regimes (agrupado exclui via numeric_only=True;
    # individual traz todas via merge_asof).
    return [c for c in df.columns
            if c not in proibidas
            and pd.api.types.is_numeric_dtype(df[c])]

def preparar_par_ts(df, classe_pos, classe_neg, pos_label, cols_excluir, corte_ts):
    """Par binário (neg vs pos) com split por TIMESTAMP fixo (request_dt), reutilizável
    entre regimes. Requer 'dataset_type' (falha cedo e claro se ausente)."""
    if 'dataset_type' not in df.columns:
        raise KeyError("base sem 'dataset_type' — use as bases em nível de evento.")
    d = df[df['dataset_type'].isin([classe_neg, classe_pos])].copy()
    d = d.sort_values('request_dt').reset_index(drop=True)
    d['y'] = (d['dataset_type'] == classe_pos).astype(int)
    feats = selecionar_features(d, cols_excluir)
    tr = d[d['request_dt'] <  corte_ts]
    te = d[d['request_dt'] >= corte_ts]
    return dict(Xtr=tr[feats].copy(), ytr=tr['y'].values,
                Xte=te[feats].copy(), yte=te['y'].values,
                feats=feats, pos_label=pos_label)

def treinar_lgbm_par(dados, titulo, verbose=True, scale_pos_weight=None):
    """LGBM binário: early stopping em sub-fatia temporal, refit em todo o treino (R3),
    reporta macro-F1, F1 por classe, PR-AUC. Se scale_pos_weight for informado, amplifica a
    classe positiva (recall ↑ / precisão ↓) — aplicado ao probe E ao modelo final."""
    Xtr, ytr = dados['Xtr'], dados['ytr']
    n = len(Xtr); k = max(1, int(n * 0.80))
    params = dict(objective='binary', n_estimators=3000, learning_rate=0.05,
                  num_leaves=63, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                  reg_lambda=1.0, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    if scale_pos_weight is not None:
        params['scale_pos_weight'] = float(scale_pos_weight)   # amplifica o sinal da classe positiva
    # Guarda para fatias degeneradas: só faz early stopping se a sub-fatia de validação
    # (Xtr.iloc[k:]) e a de treino (Xtr.iloc[:k]) forem não-vazias e o treino tiver ≥2 linhas.
    # (Inatingível na amostra balanceada real, mas evita crash do LightGBM em par minúsculo.)
    if n >= 2 and 0 < k < n:
        probe = LGBMClassifier(**params)
        probe.fit(Xtr.iloc[:k], ytr[:k], eval_set=[(Xtr.iloc[k:], ytr[k:])],
                  eval_metric='binary_logloss',
                  callbacks=[early_stopping(40), log_evaluation(0)])
        best_iter = int(probe.best_iteration_ or params['n_estimators'])
    else:
        print(f'AVISO: par degenerado (n={n}); pulando early stopping (usa n_estimators=300).')
        best_iter = 300
    model = LGBMClassifier(**{**params, 'n_estimators': best_iter})
    model.fit(Xtr, ytr)   # R3: refit em todo o treino
    proba = model.predict_proba(dados['Xte'])[:, 1]
    pred  = (proba >= 0.5).astype(int)
    macro = f1_score(dados['yte'], pred, average='macro')
    prauc = average_precision_score(dados['yte'], proba)
    if verbose:
        print(f'=== {titulo} ===')
        print(classification_report(dados['yte'], pred, digits=3,
                                    target_names=['NAO_EVENTO', dados['pos_label']]))
        print(f'macro-F1={macro:.4f} | PR-AUC={prauc:.4f} | best_iter={best_iter}'
              + ('' if scale_pos_weight is None else f' | scale_pos_weight={scale_pos_weight}'))
    return dict(model=model, macro_f1=macro, pr_auc=prauc, best_iter=best_iter,
                feats=dados['feats'])

def comparar_regimes(base_agr, base_ind, classe_pos, pos_label, cols_excluir, corte_ts):
    dA = preparar_par_ts(base_agr, classe_pos, 'DS_OUTROS', pos_label, cols_excluir, corte_ts)
    rA = treinar_lgbm_par(dA, f'{pos_label} — AGRUPADO ({JANELA_AGRUPADO})')
    dI = preparar_par_ts(base_ind, classe_pos, 'DS_OUTROS', pos_label, cols_excluir, corte_ts)
    rI = treinar_lgbm_par(dI, f'{pos_label} — INDIVIDUAL (instantâneo)')
    tab = pd.DataFrame({'regime': ['agrupado', 'individual'],
                        'macro_f1': [rA['macro_f1'], rI['macro_f1']],
                        'pr_auc':   [rA['pr_auc'],   rI['pr_auc']]})
    print('\n', tab.to_string(index=False))
    return {'agrupado': rA, 'individual': rI}, tab


In [ ]:
# Bases do ESTUDO + corte temporal comum (por request_dt, idêntico entre regimes)
base_agr = construir_base_agrupado_estudo(JANELA_AGRUPADO)
base_ind = construir_base_individual_estudo(tol='1h')
CORTE_TS = pd.concat([base_agr['request_dt'], base_ind['request_dt']]).quantile(0.80)
print('CORTE_TS =', CORTE_TS, '| base_agr', base_agr.shape, '| base_ind', base_ind.shape)


## 5. Análise A — Não-Evento vs. Clima (individual vs. agrupado)


In [ ]:
cols_excluir_A = sorted(set(
    [c for c in base_agr.columns if eh_score_composto(c)] +
    [c for c in base_ind.columns if eh_score_composto(c)] +
    [c for c in base_agr.columns if eh_vazamento_atraso(c)] +
    [c for c in base_ind.columns if eh_vazamento_atraso(c)]))
print(f'Análise A: {len(cols_excluir_A)} colunas excluídas (escore composto + voos/atraso).')
resA, tabA = comparar_regimes(base_agr, base_ind, 'DS_CLIMA', 'CLIMA',
                              cols_excluir_A, CORTE_TS)


In [ ]:
# SHAP do regime vencedor da Análise A (maior PR-AUC)
t = tabA.set_index('regime')
vencedor = 'agrupado' if t.loc['agrupado', 'pr_auc'] >= t.loc['individual', 'pr_auc'] else 'individual'
baseV = base_agr if vencedor == 'agrupado' else base_ind
dSh = preparar_par_ts(baseV, 'DS_CLIMA', 'DS_OUTROS', 'CLIMA', cols_excluir_A, CORTE_TS)
exp = shap.TreeExplainer(resA[vencedor]['model'])
Xsh = dSh['Xte'].iloc[:min(3000, len(dSh['Xte']))]
sv = exp.shap_values(Xsh)
sv1 = sv[1] if isinstance(sv, list) else (sv[..., 1] if getattr(sv, 'ndim', 2) == 3 else sv)
shap.summary_plot(sv1, Xsh, plot_type='bar', show=True)
print('Regime vencedor (Análise A):', vencedor)


## 6. Análise B — Não-Evento vs. Atraso (guarda anti-vazamento)

O CSV de voos contém **apenas voos atrasados**, então `qtd_voos`, `voos_*`, `atraso_*` e
`sev_*` são **pós-fato** para prever atraso. `eh_vazamento_atraso` exclui todos eles. A
tarefa fica genuinamente preditiva (clima + tempo). A comparação individual×agrupado é
limpa só para o clima (Análise A); B é reportada no regime agrupado.


In [ ]:
cols_excluir_B = sorted(set(
    [c for c in base_agr.columns if eh_score_composto(c)] +
    [c for c in base_agr.columns if eh_vazamento_atraso(c)]))
print(f'Análise B: {len(cols_excluir_B)} colunas excluídas.')
dB = preparar_par_ts(base_agr, 'DS_VOO', 'DS_OUTROS', 'ATRASO', cols_excluir_B, CORTE_TS)
print('Features usadas na Análise B (deve ser clima+tempo, sem voos):', dB['feats'])
resB = treinar_lgbm_par(dB, 'ATRASO — AGRUPADO (sem vazamento pós-fato)')


## 7. Descritivas (contexto)


In [ ]:
ev = pd.concat([DS_CLIMA.assign(tipo='CLIMA'), DS_VOO.assign(tipo='VOO')], ignore_index=True)
ev['hora'] = ev['request_dt'].dt.hour; ev['dia'] = ev['request_dt'].dt.dayofweek
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ev.groupby(['hora', 'tipo']).size().unstack(fill_value=0).plot(ax=ax[0])
ax[0].set_title('Eventos por hora do dia')
ev.groupby(['dia', 'tipo']).size().unstack(fill_value=0).plot.bar(ax=ax[1])
ax[1].set_title('Eventos por dia da semana')
plt.tight_layout(); plt.show()


## 8. EXPORTAÇÃO DE PRODUÇÃO — modelos pareados para o Narrador (05_5)

As células a seguir treinam os dois modelos pareados de produção (clima e atraso),
agregam os voos por janela, anexam a **climatologia prospectiva de atraso** (prior
por `dia_semana` x `hora`, sem vazamento) e exportam tudo no contrato consumido pelo
`05_5_Narrador_Natural.ipynb`.


In [ ]:
def _agregar_voos_prod(dfv, window_size):
    """IDÊNTICO ao _agregar_voos do 05_5: features enriquecidas de voos por janela."""
    dfv = dfv.copy()
    # coluna de tempo do voo (CHEGADA_REAL) -> janela
    if 'CHEGADA_REAL' not in dfv.columns:
        raise KeyError("_agregar_voos_prod: coluna 'CHEGADA_REAL' ausente no CSV de voos — "
                       "sem ela não há como janelar os voos. Verifique o schema de df_voos.")
    dfv['CHEGADA_REAL'] = pd.to_datetime(dfv['CHEGADA_REAL'], errors='coerce', utc=True)
    dfv['time_window'] = dfv['CHEGADA_REAL'].dt.floor(window_size)
    dfv = dfv.dropna(subset=['time_window'])
    if 'CODIGO_TIPO_LINHA' in dfv.columns:
        lt = dfv['CODIGO_TIPO_LINHA'].astype(str).str.strip().str.upper()
        dfv['linha_N'] = (lt == 'N').astype(int); dfv['linha_I'] = (lt == 'I').astype(int)
    if 'ICAO_EMPRESA_AEREA' in dfv.columns:
        cia = dfv['ICAO_EMPRESA_AEREA'].astype(str).str.upper()
        for c in TOP_CIAS: dfv[f'cia_{c}'] = (cia == c).astype(int)
    if 'ICAO_AERODROMO_ORIGEM' in dfv.columns:
        org = dfv['ICAO_AERODROMO_ORIGEM'].astype(str).str.upper()
        for o in TOP_ORIGENS: dfv[f'orig_{o}'] = (org == o).astype(int)
    if 'STATUS_ATRASO' in dfv.columns:
        st = dfv['STATUS_ATRASO'].astype(str)
        dfv['sev_leve']     = st.str.contains('Leve', case=False, na=False).astype(int)
        dfv['sev_moderado'] = st.str.contains('Moderado', case=False, na=False).astype(int)
        dfv['sev_grave']    = st.str.contains('Grave', case=False, na=False).astype(int)
    agg = {'qtd_voos_previstos': ('time_window', 'size')}
    if 'ICAO_EMPRESA_AEREA' in dfv.columns:
        agg['qtd_empresas_aereas'] = ('ICAO_EMPRESA_AEREA', 'nunique')
    if 'ATRASO_MINUTOS' in dfv.columns:
        dfv['ATRASO_MINUTOS'] = pd.to_numeric(dfv['ATRASO_MINUTOS'], errors='coerce')
        agg['atraso_medio_min'] = ('ATRASO_MINUTOS', 'mean')
        agg['atraso_max_min']   = ('ATRASO_MINUTOS', 'max')
    ind_cols = [c for c in dfv.columns if c.startswith(('linha_', 'cia_', 'orig_', 'sev_'))]
    for c in ind_cols:
        agg[f'voos_{c}'] = (c, 'sum')
    return dfv.groupby('time_window').agg(**agg).reset_index()

def _h3_to_latlng(hcell):
    if h3 is None:
        return AER_LAT, AER_LNG
    try:
        return h3.cell_to_latlng(hcell)
    except AttributeError:
        return h3.h3_to_geo(hcell)

def _haversine_km(lat1, lng1, lat2, lng2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1); dl = np.radians(lng2 - lng1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

def construir_base_producao():
    """Base de PRODUÇÃO (1h), namespace do 05_5. Clima agregado por janela (sem rolling
    multi-horizonte) + lags/diffs das 4 variáveis do contrato + voos enriquecidos +
    temporal + prior de atraso (climatologia + reputação, prospectivo) + dist opcional.
    Em nível de EVENTO (dataset_type/request_dt).

    O CORTE_PROD (80/20 temporal) é calculado ANTES do prior, e o prior é construído
    train-only (voos com CHEGADA_PREVISTA <= CORTE_PROD): isso torna a avaliação honesta
    (voos de teste não vazam) e, como o modelo exportado também só treina nesse período,
    o mesmo prior serve treino e produção sem skew. Define os globais CORTE_PROD, PRIOR_DF,
    PRIOR_FALLBACK e REP_CTX (usados na exportação e no meta)."""
    global CORTE_PROD, PRIOR_DF, PRIOR_FALLBACK, REP_CTX
    w = JANELA_PROD
    # clima por janela (colunas cruas do CSV; SEM enriquecimento multi-horizonte)
    c = df_clima.copy(); c['time_window'] = c['time'].dt.floor(w)
    clima_agg = c.drop(columns=['time']).groupby('time_window').mean(numeric_only=True).sort_index()
    for col in COLS_LAG_PROD:
        if col in clima_agg.columns:
            clima_agg[f'{col}_lag']  = clima_agg[col].shift(1)
            clima_agg[f'diff_{col}'] = clima_agg[col] - clima_agg[col].shift(1)
    clima_agg = clima_agg.reset_index()
    voos_agg = _agregar_voos_prod(df_voos, w)

    ev = pd.concat([DS_OUTROS, DS_CLIMA, DS_VOO], ignore_index=True) \
           .sort_values('request_dt').reset_index(drop=True)
    ev['time_window'] = ev['request_dt'].dt.floor(w)
    # Corte temporal de produção (80% treino / 20% teste), calculado ANTES do prior para que
    # o prior seja ajustado somente com voos do período de treino (sem vazamento temporal).
    CORTE_PROD = ev['request_dt'].quantile(0.80)
    PRIOR_DF, PRIOR_FALLBACK, REP_CTX = construir_prior(CONSOLIDADO_VOOS, corte_utc=CORTE_PROD)

    base = ev.merge(clima_agg, on='time_window', how='left') \
             .merge(voos_agg, on='time_window', how='left')
    # voos ausentes na janela -> 0 (janela sem voo atrasado)
    for col in base.columns:
        if col.startswith('voos_') or col in ('qtd_voos_previstos', 'qtd_empresas_aereas',
                                              'atraso_medio_min', 'atraso_max_min'):
            base[col] = base[col].fillna(0.0)
    _add_temporais(base, 'request_dt')
    # Prior de atraso (climatologia + reputação por dia_semana x hora, em UTC): junta pelos
    # campos temporais já criados por _add_temporais. Ausências -> fallback global por coluna
    # (raro: só (dia_semana,hora) sem nenhum voo histórico). Estas features NÃO são pós-fato.
    base = base.merge(PRIOR_DF, on=['dia_semana', 'hora'], how='left')
    for col in COLS_CLIMATOLOGIA:
        if col in base.columns:
            base[col] = base[col].fillna(PRIOR_FALLBACK[col])
    if USE_DIST and 'origin_h3' in base.columns:
        coords = base['origin_h3'].map(lambda hh: _h3_to_latlng(hh) if pd.notna(hh) else (AER_LAT, AER_LNG))
        base['lat'] = coords.map(lambda t: t[0]); base['lng'] = coords.map(lambda t: t[1])
        base['dist_aeroporto_km'] = _haversine_km(base['lat'], base['lng'], AER_LAT, AER_LNG)
    return base

base_prod = construir_base_producao()
PRIOR_DF.to_csv(EXP_CLIMATOLOGIA, index=False, encoding='utf-8')      # <- o 05_5 lê o prior no serve
REP_CTX.to_csv(EXP_REPUTACAO_SLOT, index=False, encoding='utf-8')     # <- contexto textual p/ a narração
_na = int(base_prod[[c for c in COLS_CLIMATOLOGIA if c in base_prod.columns]].isna().sum().sum())
print('base_prod', base_prod.shape, '| CORTE_PROD', CORTE_PROD, '| NaN no prior:', _na)
print('prior (%d slots) -> %s' % (len(PRIOR_DF), EXP_CLIMATOLOGIA))
print('contexto reputação (%d linhas) -> %s' % (len(REP_CTX), EXP_REPUTACAO_SLOT))

In [ ]:
# ------- Seleção de features de PRODUÇÃO para cada modelo pareado -------
COLS_VOOS_PROD = ([c for c in base_prod.columns if c.startswith('voos_')] +
                  ['qtd_voos_previstos', 'qtd_empresas_aereas', 'atraso_medio_min', 'atraso_max_min'])
COLS_VOOS_PROD = [c for c in COLS_VOOS_PROD if c in base_prod.columns]

# CLIMA: só variáveis físicas/térmicas (escores compostos fora) + temporal (+dist). SEM voos
# e SEM a climatologia de atraso (o modelo de clima é sobre tempo, não sobre atraso de voo).
cols_excluir_clima_prod = sorted(set(
    [c for c in base_prod.columns if eh_score_composto(c)] + COLS_VOOS_PROD + COLS_CLIMATOLOGIA))
# ATRASO: escores fora sempre; voos PÓS-FATO fora por padrão (flag). Temporal + clima +
# climatologia (+dist). ATENÇÃO: COLS_CLIMATOLOGIA (taxa_atraso_hist, ...) são PROSPECTIVAS
# (prior por dia_semana x hora, disponíveis antes do fato) -> ficam DENTRO do modelo de
# atraso de propósito, e por isso NÃO entram na lista de exclusão abaixo.
cols_excluir_atraso_prod = sorted(set([c for c in base_prod.columns if eh_score_composto(c)]))
if not INCLUIR_FEATURES_POSFATO_ATRASO:
    cols_excluir_atraso_prod = sorted(set(cols_excluir_atraso_prod + COLS_VOOS_PROD))

def treinar_e_exportar_par(classe_pos, cols_excluir, model_path, meta_path, rotulo,
                           scale_pos_weight=None):
    d = preparar_par_ts(base_prod, classe_pos, 'DS_OUTROS', rotulo, cols_excluir, CORTE_PROD)
    r = treinar_lgbm_par(d, f'PRODUÇÃO {rotulo}', scale_pos_weight=scale_pos_weight)
    meta = {
        'janela': JANELA_PROD,
        'dist_aeroporto': bool(USE_DIST),
        'scale_pos_weight': scale_pos_weight,   # None (clima) ou PESO_ATRASO (atraso)
        'features_lgbm': list(r['feats']),
        'macro_f1_bal': round(float(r['macro_f1']), 4),
        'macro_f1_nat': round(float(r['macro_f1']), 4),   # teste balanceado (ver obs/L2)
        'pr_auc': round(float(r['pr_auc']), 4),
        'best_iteration': int(r['best_iter']),
        'posfato_atraso': bool(INCLUIR_FEATURES_POSFATO_ATRASO),
        'climatologia_atraso': [c for c in COLS_CLIMATOLOGIA if c in r['feats']],
        'climatologia_fallback': {col: float(PRIOR_FALLBACK[col]) for col in COLS_CLIMATOLOGIA},  # fallback por coluna (sem skew)
        'taxa_atraso_global': float(PRIOR_FALLBACK['taxa_atraso_hist']),   # retrocompat (fallback antigo do 05_5)
        'obs': ('Teste balanceado (query equaliza classes); macro_f1_nat=bal pois nao ha '
                'conjunto natural. Ver limitacoes L1/L2 no .txt.'),
    }
    import joblib
    joblib.dump(r['model'], model_path)
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    print(f'Exportado: {model_path}\n  features_lgbm ({len(meta["features_lgbm"])}):',
          meta['features_lgbm'])
    return r, meta

res_clima_prod, meta_clima  = treinar_e_exportar_par(
    'DS_CLIMA', cols_excluir_clima_prod, EXP_CLIMA, EXP_CLIMA_M, 'CLIMA')
res_atraso_prod, meta_atraso = treinar_e_exportar_par(
    'DS_VOO', cols_excluir_atraso_prod, EXP_ATRASO, EXP_ATRASO_M, 'ATRASO',
    scale_pos_weight=PESO_ATRASO)

In [ ]:
# ------- Verificação de compatibilidade com o narrador (05_5) -------
# O 05_5.construir_features() gera este superconjunto de nomes. Toda feature_lgbm exportada
# deve pertencer a ele; caso contrário o narrador a preencheria com 0 (train/serve skew).
NAMESPACE_NARRADOR = set(
    list(df_clima.select_dtypes('number').columns) +                       # clima cru
    [f'{c}_lag' for c in COLS_LAG_PROD] + [f'diff_{c}' for c in COLS_LAG_PROD] +
    ['qtd_voos_previstos', 'qtd_empresas_aereas', 'atraso_medio_min', 'atraso_max_min'] +
    [f'voos_cia_{c}' for c in TOP_CIAS] + [f'voos_orig_{o}' for o in TOP_ORIGENS] +
    ['voos_linha_N', 'voos_linha_I', 'voos_sev_leve', 'voos_sev_moderado', 'voos_sev_grave'] +
    COLS_CLIMATOLOGIA +                                                     # climatologia de atraso
    ['lat', 'lng', 'dist_aeroporto_km', 'hora', 'mes', 'dia_semana',
     'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos', 'dia_semana_sin', 'dia_semana_cos'])

for nome, meta in [('clima', meta_clima), ('atraso', meta_atraso)]:
    fora = [f for f in meta['features_lgbm'] if f not in NAMESPACE_NARRADOR]
    if fora:
        print(f"ATENÇÃO [{nome}]: features fora do namespace do 05_5 (o narrador as zeraria): {fora}")
    else:
        print(f"OK [{nome}]: todas as {len(meta['features_lgbm'])} features são compatíveis com o 05_5.")

# AÇÃO NECESSÁRIA no 05_5 (companheiro deste export): para NÃO zerar a climatologia no serve,
# o narrador precisa carregar a tabela exportada e reproduzir as 3 features por (dia_semana,
# hora) em UTC — o mesmo par temporal que ele já calcula. Enquanto isso não for feito, o
# 05_5 preencherá taxa_atraso_hist/atraso_medio_hist/chegadas_previstas_hist com 0 (skew).
if any(f in meta_atraso['features_lgbm'] for f in COLS_CLIMATOLOGIA):
    print(f"\n[05_5] Carregar a climatologia exportada em '{EXP_CLIMATOLOGIA}' e reconstruir "
          f"{[f for f in COLS_CLIMATOLOGIA if f in meta_atraso['features_lgbm']]} no serve.")
print('\nModelos prontos para o 05_5_Narrador_Natural.ipynb.')

## 9. Conclusões e limitações (honestas)

**Resultados.** `tabA` (Análise A: individual vs. agrupado) e `resB` (Análise B) trazem
macro-F1 e PR-AUC. Priorize **PR-AUC** sob desbalanceamento. Os modelos de produção
(Seção 8) foram exportados no contrato do `05_5`.

**Limitações (para a tese):**
1. **Granularidade só é comparável acima da resolução do clima** — daí a janela de 4h no
   estudo; a produção usa 1h (onde agrupado≡individual, correto para operação).
2. **Teste não é populacional** (query equaliza classes) — métricas são pareadas/balanceadas.
   Próximo passo natural: avaliar em prevalência real (taxa base de atraso na chegada em SBPA
   ≈ 15,6% em 2025) e calibrar o limiar de decisão.
3. **Modelo de atraso: pós-fato vs. prospectivo.** As features derivadas do
   `03_voos_atrasados_sbpa.csv` (`voos_*`, `qtd_voos`, `atraso_*`) continuam **pós-fato** e
   ficam fora por padrão (`INCLUIR_FEATURES_POSFATO_ATRASO=False`). **Novidade:** a partir do
   VRA consolidado (`01_todos_voos_consolidados.csv`, que preserva os voos pontuais = o
   denominador), o modelo de atraso agora recebe uma **climatologia PROSPECTIVA** por
   `(dia_semana, hora)` — `taxa_atraso_hist`, `atraso_medio_hist`, `chegadas_previstas_hist` —
   que é um prior conhecido **antes** do fato e, portanto, **não vaza**. É o primeiro sinal de
   atraso legítimo (não-climático) que entra no modelo.
   - *Ressalva de rigor:* a climatologia é agregada sobre o ano inteiro (prior estável); a
     contribuição de qualquer janela individual ao seu próprio agregado é ~1/52, desprezível.
     Para rigor máximo dá para computá-la só no período de treino (`< CORTE_PROD`).
   - *Fuso:* a chave usa `CHEGADA_PREVISTA` convertida de `TZ_VRA` para UTC. Se os eventos
     tiverem sido ancorados tratando o horário do VRA como UTC-ingênuo, use `TZ_VRA=None`.
   - *Serve (05_5):* exige o companheiro — o narrador precisa carregar
     `climatologia_atraso_sbpa.csv` e reconstruir as 3 features, senão elas são zeradas.
4. **Vazamento espacial** — `lat/lng/densidade_h3` ficam fora das features. A partir da
   v0.13.0 `dist_aeroporto_km` **também sai** do modelo (`USE_DIST=False`): antes era servida
   como constante fixa (SHAP enganoso, sem variação); agora é tratada só como **esforço do
   motorista** no 05_5 (ajusta a barra de decisão, sem entrar no modelo/SHAP).
5. **Amplificação do sinal de atraso** — o modelo de atraso passa a treinar com
   `scale_pos_weight = PESO_ATRASO` (>1): aumenta o **recall** da classe "atraso" para captar
   padrões concentrados mesmo esporádicos, ao custo de alguma precisão. Verifique no
   `classification_report` que o recall de ATRASO sobe vs. o baseline sem peso.